# تحلیل اکتشافی داده‌های اضطراب اجتماعی

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

pd.set_option("display.max_columns", None)
px.defaults.template = "plotly_white"

## ۱. درک مسئله و سوالات تحلیل

## ۲. نگاه اولیه

In [ ]:
def load_data(path):
    """Load the raw dataset.

    Parameters
    ----------
    path : str
        Path to the CSV file.

    Returns
    -------
    pandas.DataFrame
    """
    return pd.read_csv(path)


def overview(df):
    """Summarize shape, dtypes, missing values and basic statistics.

    Parameters
    ----------
    df : pandas.DataFrame

    Returns
    -------
    pandas.DataFrame
        One row per column: dtype, missing count, missing %, unique count, min, max.
    """
    return pd.DataFrame({
        "dtype": df.dtypes,
        "missing": df.isna().sum(),
        "missing_%": (df.isna().mean() * 100).round(1),
        "unique": df.nunique(),
        "min": df.min(numeric_only=True),
        "max": df.max(numeric_only=True),
    })

In [ ]:
df = load_data("social_anxiety_dataset.csv")
overview(df)

## ۳. تحلیل تک‌متغیره

In [ ]:
def plot_numeric(df, col, title=None):
    """Histogram, box plot and violin plot of a numeric column side by side.

    Parameters
    ----------
    df : pandas.DataFrame
    col : str
        Numeric column name.
    title : str, optional
        Figure title. Defaults to the column name.

    Returns
    -------
    plotly.graph_objects.Figure
    """
    fig = make_subplots(rows=1, cols=3, subplot_titles=("Histogram", "Box", "Violin"))
    fig.add_trace(go.Histogram(x=df[col], name="hist"), row=1, col=1)
    fig.add_trace(go.Box(y=df[col], name="box"), row=1, col=2)
    fig.add_trace(go.Violin(y=df[col], name="violin", box_visible=True), row=1, col=3)
    fig.update_layout(title=title or col, showlegend=False, height=400)
    return fig


def plot_categorical(df, col, title=None):
    """Bar chart of category frequencies.

    Parameters
    ----------
    df : pandas.DataFrame
    col : str
        Categorical column name.
    title : str, optional

    Returns
    -------
    plotly.graph_objects.Figure
    """
    counts = df[col].value_counts(dropna=False).reset_index()
    counts.columns = [col, "count"]
    return px.bar(counts, x=col, y="count", text="count", title=title or col)


def describe_numeric(df, col):
    """Descriptive statistics of a numeric column.

    Parameters
    ----------
    df : pandas.DataFrame
    col : str

    Returns
    -------
    pandas.Series
        count, mean, median, std, skew, min, q1, q3, max.
    """
    s = df[col].dropna()
    return pd.Series({
        "count": s.count(), "mean": s.mean(), "median": s.median(), "std": s.std(),
        "skew": s.skew(), "min": s.min(), "q1": s.quantile(0.25), "q3": s.quantile(0.75), "max": s.max(),
    })

### ۳.۱ جمعیت‌شناختی

### ۳.۲ سبک زندگی

### ۳.۳ فیزیولوژیک

### ۳.۴ درمان و سابقه

### ۳.۵ متغیر هدف

## ۴. پاک‌سازی داده

In [ ]:
def find_outliers(df, col, method="iqr", z_thresh=3.0):
    """Boolean mask of outliers in a numeric column.

    Parameters
    ----------
    df : pandas.DataFrame
    col : str
    method : {"iqr", "zscore"}
    z_thresh : float
        Threshold used when method is "zscore".

    Returns
    -------
    pandas.Series of bool
    """
    s = df[col]
    if method == "iqr":
        q1, q3 = s.quantile([0.25, 0.75])
        iqr = q3 - q1
        return (s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)
    z = (s - s.mean()) / s.std()
    return z.abs() > z_thresh


def clean_data(df):
    """Apply all agreed cleaning decisions and return the clean dataset.

    Parameters
    ----------
    df : pandas.DataFrame
        Raw dataset.

    Returns
    -------
    pandas.DataFrame
    """
    out = df.copy()
    return out

In [ ]:
clean_df = clean_data(df)
overview(clean_df)

## ۵. تحلیل دومتغیره

In [ ]:
def correlation_with_target(df, target, method="spearman"):
    """Correlation of every numeric column with the target, with p-values.

    Parameters
    ----------
    df : pandas.DataFrame
    target : str
        Numeric target column.
    method : {"pearson", "spearman"}

    Returns
    -------
    pandas.DataFrame
        Columns: r, p_value, significant. Sorted by |r| descending.
    """
    rows = []
    func = stats.pearsonr if method == "pearson" else stats.spearmanr
    for col in df.select_dtypes("number").columns.drop(target):
        pair = df[[col, target]].dropna()
        r, p = func(pair[col], pair[target])
        rows.append({"feature": col, "r": r, "p_value": p, "significant": p < 0.05})
    result = pd.DataFrame(rows).set_index("feature")
    return result.loc[result["r"].abs().sort_values(ascending=False).index]


def compare_groups(df, cat, num):
    """Compare a numeric column across the categories of a categorical column.

    Uses Welch t-test for two groups and one-way ANOVA for more.

    Parameters
    ----------
    df : pandas.DataFrame
    cat : str
        Categorical column.
    num : str
        Numeric column.

    Returns
    -------
    tuple
        (summary table of mean/median/count per group, test name, statistic, p-value)
    """
    groups = [g[num].dropna() for _, g in df.groupby(cat)]
    summary = df.groupby(cat)[num].agg(["mean", "median", "count"])
    if len(groups) == 2:
        stat, p = stats.ttest_ind(*groups, equal_var=False)
        return summary, "welch_t", stat, p
    stat, p = stats.f_oneway(*groups)
    return summary, "anova", stat, p


def crosstab_test(df, a, b):
    """Contingency table of two categorical columns with chi-square test.

    Parameters
    ----------
    df : pandas.DataFrame
    a, b : str
        Categorical columns.

    Returns
    -------
    tuple
        (crosstab, chi2, p-value, Cramer's V)
    """
    ct = pd.crosstab(df[a], df[b])
    chi2, p, _, _ = stats.chi2_contingency(ct)
    n = ct.values.sum()
    v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
    return ct, chi2, p, v

### ۵.۱ عددی با عددی

### ۵.۲ دسته‌ای با عددی

### ۵.۳ دسته‌ای با دسته‌ای

## ۶. تحلیل چندمتغیره و KPI

In [ ]:
def add_features(df):
    """Add derived KPI and interaction columns.

    Parameters
    ----------
    df : pandas.DataFrame
        Clean dataset.

    Returns
    -------
    pandas.DataFrame
    """
    out = df.copy()
    return out


def pivot_heatmap(df, index, columns, values, aggfunc="mean", title=None):
    """Heatmap of an aggregated value across two categorical columns.

    Parameters
    ----------
    df : pandas.DataFrame
    index, columns : str
        Categorical columns for rows and columns.
    values : str
        Numeric column to aggregate.
    aggfunc : str or callable
    title : str, optional

    Returns
    -------
    plotly.graph_objects.Figure
    """
    table = df.pivot_table(index=index, columns=columns, values=values, aggfunc=aggfunc)
    return px.imshow(table, text_auto=".2f", aspect="auto", title=title or f"{aggfunc} {values}")

## ۷. یافته‌های کلیدی